# Clean and Join

Maps postcode-level transactions to local authorities and creates LA×month summaries for hedonic indexing.


## Setup

In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

RAW = Path('../data/raw')
INTERIM = Path('../data/interim')
PROCESSED = Path('../data/processed')
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Interim path:', INTERIM.resolve())

Project root: /Users/umar/Documents/Northeastern/Dissertation
Interim path: /Users/umar/Documents/Northeastern/Dissertation/property/data/interim


## Load Raw Data

In [7]:
ppd_raw_path = INTERIM / 'ppd_raw.parquet'
onspd_path   = INTERIM / 'onspd_postcode_to_la.parquet'
la_path      = INTERIM / 'la_boundaries.parquet'
assert ppd_raw_path.exists(), f'Missing {ppd_raw_path}'
assert onspd_path.exists(), f'Missing {onspd_path}'
assert la_path.exists(), f'Missing {la_path}'

ppd_raw = pd.read_parquet(ppd_raw_path)
onspd = pd.read_parquet(onspd_path)
la = pd.read_parquet(la_path)[['LA_code','LA_name']]
print('ppd_raw shape:', ppd_raw.shape)
print('onspd shape:', onspd.shape)
ppd_raw.head()

ppd_raw shape: (30630140, 6)
onspd shape: (2403580, 2)


,price,date,postcode,property_type,old_new,duration
0,70000,1995-07-07,MK15 9HP,D,N,F
1,44500,1995-02-03,SR6 0AQ,T,N,F
2,56500,1995-01-13,CO6 1SQ,T,N,F
3,58000,1995-07-28,B90 4TG,T,N,F
4,51000,1995-06-28,DY5 1SA,S,N,F


## Clean Postcodes and Attach LA Codes

In [10]:
ppd = ppd_raw.copy()
ppd['postcode'] = (ppd['postcode'].astype(str).str.upper().str.replace(' ', '', regex=False))
onspd['postcode'] = onspd['postcode'].astype(str).str.upper().str.replace(' ', '', regex=False)

ppd = ppd.merge(onspd, on='postcode', how='left', validate='many_to_one', indicator=True)
match_rate = ppd['LA_code'].notna().mean() * 100
print(f'LA match rate: {match_rate:.2f}%')

# see at a few missing postcodes for later diagnosis
if match_rate < 98:
    sample_miss = (ppd.loc[ppd['LA_code'].isna(), ['postcode']]
                     .drop_duplicates()
                     .head(20))
    print('Sample missing postcodes:\n', sample_miss)

ppd = ppd.dropna(subset=['LA_code']).drop(columns=['_merge'])
ppd.head()

LA match rate: 99.83%


,price,date,postcode,property_type,old_new,duration,LA_code
0,70000,1995-07-07,MK159HP,D,N,F,E06000042
1,44500,1995-02-03,SR60AQ,T,N,F,E08000024
2,56500,1995-01-13,CO61SQ,T,N,F,E07000067
3,58000,1995-07-28,B904TG,T,N,F,E08000029
4,51000,1995-06-28,DY51SA,S,N,F,E08000027


## Filter Prices and Parse Dates

In [13]:
ppd['price'] = pd.to_numeric(ppd['price'], errors='coerce')
ppd['date'] = pd.to_datetime(ppd['date'], errors='coerce')
ppd = ppd.dropna(subset=['price', 'date'])
ppd = ppd[(ppd['price'] >= 10_000) & (ppd['price'] <= 5_000_000)]
ppd['month'] = ppd['date'].dt.to_period('M').dt.to_timestamp('M')
ppd['log_price'] = np.log(ppd['price'])
print('After filters:', ppd.shape)

After filters: (30491147, 9)


## Add LA Names and Save

In [16]:
# Add LA names 
ppd = ppd.merge(la, on='LA_code', how='left')

# keep standard duration values only 
ppd['duration'] = ppd['duration'].astype(str).str.upper()
ppd = ppd[ppd['duration'].isin(['F','L'])]

# Consistent categoricals (helps RAM + downstream grouping) 
ppd['property_type'] = pd.Categorical(
    ppd['property_type'].astype(str).str.upper(),
    categories=['D','S','T','F','O']  # Detached, Semi, Terraced, Flat, Other
)
ppd['old_new'] = pd.Categorical(
    ppd['old_new'].astype(str).str.upper(),
    categories=['Y','N']  # New build, Not new
)
ppd['duration'] = pd.Categorical(ppd['duration'], categories=['F','L'])

# Dedupe (PPD can have silent duplicates after joins) 
dedupe_cols = ['date','postcode','price','property_type','old_new','duration']
ppd = ppd.drop_duplicates(subset=dedupe_cols)

# Slim transaction fact table 
ppd_clean_cols = [
    'date','month','LA_code','LA_name','price','log_price',
    'postcode','property_type','old_new','duration'
]
ppd_clean = ppd[ppd_clean_cols].reset_index(drop=True)

# Save 
ppd_clean_path = INTERIM / 'ppd_la_month_transactions.parquet'
ppd_clean.to_parquet(ppd_clean_path, index=False, compression='zstd')
print('Saved cleaned transactions', ppd_clean_path, f'({len(ppd_clean):,} rows)')
ppd_clean.head()


Saved cleaned transactions ../data/interim/ppd_la_month_transactions.parquet (30,218,056 rows)


,date,month,LA_code,LA_name,price,log_price,postcode,property_type,old_new,duration
0,1995-07-07,1995-07-31,E06000042,Milton Keynes,70000,11.156251,MK159HP,D,N,F
1,1995-02-03,1995-02-28,E08000024,Sunderland,44500,10.703244,SR60AQ,T,N,F
2,1995-01-13,1995-01-31,E07000067,Braintree,56500,10.941996,CO61SQ,T,N,F
3,1995-07-28,1995-07-31,E08000029,Solihull,58000,10.968198,B904TG,T,N,F
4,1995-06-28,1995-06-30,E08000027,Dudley,51000,10.839581,DY51SA,S,N,F


## Aggregate to LA×Month

In [19]:
la_month = (
    ppd_clean
    .groupby(['LA_code','LA_name','month'], as_index=False, sort=False, observed=True)
    .agg(
        median_price=('price','median'),
        mean_price=('price','mean'),
        n_sales=('price','size')
    )
)
la_month['is_thin'] = la_month['n_sales'] < 20  

la_month_path = INTERIM / 'la_month_prices.parquet'
la_month.to_parquet(la_month_path, index=False, compression='zstd')
print('Saved LA×month summary', la_month_path, f'({len(la_month):,} rows)')
la_month.head()


Saved LA×month summary ../data/interim/la_month_prices.parquet (117,249 rows)


,LA_code,LA_name,month,median_price,mean_price,n_sales,is_thin
0,E06000042,Milton Keynes,1995-07-31,53000.0,63324.771341,328,False
1,E08000024,Sunderland,1995-02-28,36950.0,43039.394089,203,False
2,E07000067,Braintree,1995-01-31,55000.0,66403.196581,117,False
3,E08000029,Solihull,1995-07-31,72450.0,88752.139918,243,False
4,E08000027,Dudley,1995-06-30,47000.0,53440.277916,403,False


## Diagnostics

In [22]:
print('Transactions after cleaning:', len(ppd_clean))
print('Unique LA codes:', ppd_clean['LA_code'].nunique())
print('\nLA-month summary describe:')
print(la_month.describe(include='all').transpose())

# Quick coverage checks / assertions
try:
    # match_rate was computed in Cell 2; guard in case it isn't in scope
    mr = float(match_rate)
except Exception:
    mr = np.nan

n_las   = la_month['LA_code'].nunique()
n_month = la_month['month'].nunique()
mon_seq = pd.date_range(la_month['month'].min(), la_month['month'].max(), freq='M')
thin_share = la_month['is_thin'].mean()

print('\nCoverage summary:')
print(f'  LA match rate:          {mr:.2f}%')
print(f'  LA count:               {n_las}')
print(f'  Months present:         {n_month} of {len(mon_seq)} in range')
print(f'  Thin-month share:       {thin_share:.1%}')

# Fail early if something looks off 
if not np.isnan(mr):
    assert mr > 95, f'Low LA match rate: {mr:.2f}%'
assert n_las >= 300, f'Suspiciously few LAs: {n_las}'
assert n_month >= 0.90 * len(mon_seq), f'Large month coverage gap: {n_month}/{len(mon_seq)} present'
assert thin_share < 0.50, f'Over half of LA×months are thin ({thin_share:.1%}); consider lowering threshold or using rolling medians in NB3.'

print('Assertions passed.')


Transactions after cleaning: 30218056
Unique LA codes: 318

LA-month summary describe:
                 count unique            top    freq  \
LA_code         117249    318      E06000042     369   
LA_name         117249    318  Milton Keynes     369   
month           117249    NaN            NaN     NaN   
median_price  117249.0    NaN            NaN     NaN   
mean_price    117249.0    NaN            NaN     NaN   
n_sales       117249.0    NaN            NaN     NaN   
is_thin         117249      2          False  116815   

                                       mean                  min  \
LA_code                                 NaN                  NaN   
LA_name                                 NaN                  NaN   
month         2010-05-31 10:01:54.402681344  1995-01-31 00:00:00   
median_price                  188738.250603              23750.0   
mean_price                    225096.790783         27280.188679   
n_sales                           257.72549             

/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_90380/3476143667.py:15: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  mon_seq = pd.date_range(la_month['month'].min(), la_month['month'].max(), freq='M')
